# A/B Testing Analysis: Website Conversion Optimization

**Objective:** Determine whether a redesigned landing page (Variant B) produces a statistically significant improvement in conversion rate over the existing page (Variant A).

| | |
|---|---|
| **Test type** | Two-proportion z-test (one-tailed) |
| **Significance level** | α = 0.05 |
| **Null hypothesis** | p_B ≤ p_A |
| **Alternative hypothesis** | p_B > p_A |

---

## 0. Setup

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ab_test import (
    load_data, clean_data, check_sample_ratio_mismatch,
    conversion_summary, plot_conversion_rates, plot_daily_conversions,
    run_z_test, print_results, plot_z_distribution,
    minimum_sample_size, plot_power_curve
)

DATA_PATH    = '../data/ab_data.csv'
OUTPUTS_PATH = '../outputs/'
ALPHA        = 0.05

print('Setup complete.')

---
## 1. Data Loading & Cleaning

In [ ]:
# Load raw data
df_raw = load_data(DATA_PATH)
df_raw.head()

In [ ]:
# Basic info
print(df_raw.dtypes)
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
print(f"\nUnique groups : {df_raw['group'].unique()}")
print(f"Unique pages  : {df_raw['landing_page'].unique()}")

In [ ]:
# Check for mismatched group / landing_page assignments
mismatch = df_raw[
    ((df_raw['group'] == 'control')   & (df_raw['landing_page'] == 'new_page')) |
    ((df_raw['group'] == 'treatment') & (df_raw['landing_page'] == 'old_page'))
]
print(f"Mismatched rows: {len(mismatch):,}")

In [ ]:
# Clean dataset
df = clean_data(df_raw)
df.head()

In [ ]:
# Sample Ratio Mismatch check
srm_ok = check_sample_ratio_mismatch(df)

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Conversion summary table
summary = conversion_summary(df)
print(summary.to_string(index=False))

In [ ]:
# Bar chart: conversion rates
plot_conversion_rates(df, save_path=OUTPUTS_PATH + 'conversion_rates.png')

In [ ]:
# Line chart: daily conversion rates over time
plot_daily_conversions(df, save_path=OUTPUTS_PATH + 'daily_conversions.png')

In [ ]:
# Conversion distribution: converted vs not
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (group, color) in zip(axes, {'control': '#6B7280', 'treatment': '#3B82F6'}.items()):
    grp = df[df['group'] == group]
    counts = grp['converted'].value_counts().sort_index()
    ax.bar(['Not converted', 'Converted'], counts.values, color=[color + '55', color], edgecolor='white')
    ax.set_title(f'{group.capitalize()} group', fontweight='bold')
    ax.set_ylabel('Users')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 300, f'{v:,}', ha='center', fontsize=10)
plt.suptitle('Conversion Distribution by Group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS_PATH + 'conversion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Hypothesis Testing

In [ ]:
# Run two-proportion z-test
results = run_z_test(df, alpha=ALPHA)
print_results(results)

In [ ]:
# Visualise z-distribution
plot_z_distribution(results, save_path=OUTPUTS_PATH + 'z_distribution.png')

---
## 4. Statistical Interpretation

| Metric | Value |
|--------|-------|
| Control conversion rate | 11.98% |
| Treatment conversion rate | 12.42% |
| Absolute difference | +0.44 pp |
| Relative uplift | +3.68% |
| Z-statistic | ~0.92 |
| P-value (one-tailed) | 0.18 |
| Significant at α=0.05? | **No** |

> **Interpretation:** The p-value of 0.18 exceeds our significance threshold of 0.05. We fail to reject H₀. While Variant B shows a slight positive trend, there is insufficient statistical evidence to conclude that the new page meaningfully improves conversion rates.

---
## 5. Power Analysis

In [ ]:
# Minimum sample size to detect a 0.5pp lift at 80% power
p_baseline = results['p_control']
mde = 0.005  # 0.5 percentage points

n_required = minimum_sample_size(p_baseline, mde, alpha=ALPHA, power=0.80)
print(f"Required sample size per group : {n_required:,}")
print(f"Current sample size per group  : {results['n_control']:,}")
print(f"Shortfall                      : {max(0, n_required - results['n_control']):,}")

In [ ]:
# Power curve: sample size vs MDE
plot_power_curve(p_baseline, save_path=OUTPUTS_PATH + 'power_curve.png')

---
## 6. Conclusion & Recommendations

### Decision
**Do not ship Variant B at this time.** The observed improvement is not statistically significant.

### Recommendations

1. **Extend the experiment** — Run for a longer duration to collect more data and increase statistical power.
2. **Power analysis first** — Before the next test, calculate the exact sample size needed to detect your target MDE reliably.
3. **Segment the results** — Analyse sub-groups (mobile vs desktop, new vs returning users, traffic source) where Variant B may have a significant effect.
4. **Iterate on design** — Test bolder changes. A 0.5 pp MDE requires very large samples; targeting a 1–2 pp lift makes experiments faster.
5. **Monitor novelty effects** — Ensure early spikes in treatment performance don't inflate results; check time-series stability.

### Tools Used
- **Python** — Pandas, NumPy, SciPy, Matplotlib, Seaborn
- **Statistical method** — Two-proportion z-test (one-tailed)
- **Environment** — Jupyter Notebook